In [5]:
import csv, sys, os
sys.path.insert(0, "..")
from basic_client import BioreactorClient, USER, PASSWORD

client = BioreactorClient()
client.login(USER, PASSWORD)
print("Logged in successfully.")

# Pick a few recipes spanning different regions of the space,
# including the pH=8 peak you found in micro sweeps.
recipes = [
    {"T": 35.0, "pH": 7.0, "F1": 1.0, "F2": 1.0, "F3": 1.0},   # mid/baseline
    {"T": 35.0, "pH": 8.0, "F1": 2.0, "F2": 2.0, "F3": 2.0},   # observed micro peak
    {"T": 45.0, "pH": 5.0, "F1": 0.5, "F2": 0.5, "F3": 0.5},   # off-region point
]

# Fewer repeats as scale gets more expensive.
repeats = {"micro": 20, "bench": 10, "pilot": 5}

out_path = os.path.join("..", "data", "noise_bias.csv")

with open(out_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["recipe_id", "scale", "T", "pH", "F1", "F2", "F3",
                      "Y", "cost_eur", "created_at"])

    for rid, r in enumerate(recipes):
        for scale, n in repeats.items():
            for i in range(n):
                res = client.run(scale, r["T"], r["pH"], r["F1"], r["F2"], r["F3"])
                writer.writerow([rid, scale, r["T"], r["pH"], r["F1"], r["F2"], r["F3"],
                                  res["Y"], res["cost_eur"], res["created_at"]])
                print(f"recipe={rid} scale={scale} rep={i+1}/{n} Y={res['Y']:.4f}")

print(f"Done. Saved to {out_path}")

Logged in successfully.
recipe=0 scale=micro rep=1/20 Y=0.2365
recipe=0 scale=micro rep=2/20 Y=0.1942
recipe=0 scale=micro rep=3/20 Y=0.1987
recipe=0 scale=micro rep=4/20 Y=0.2142
recipe=0 scale=micro rep=5/20 Y=0.2076
recipe=0 scale=micro rep=6/20 Y=0.2460
recipe=0 scale=micro rep=7/20 Y=0.1913
recipe=0 scale=micro rep=8/20 Y=0.1832
recipe=0 scale=micro rep=9/20 Y=0.1851
recipe=0 scale=micro rep=10/20 Y=0.1818
recipe=0 scale=micro rep=11/20 Y=0.1504
recipe=0 scale=micro rep=12/20 Y=0.2142
recipe=0 scale=micro rep=13/20 Y=0.2180
recipe=0 scale=micro rep=14/20 Y=0.2038
recipe=0 scale=micro rep=15/20 Y=0.2036
recipe=0 scale=micro rep=16/20 Y=0.2196
recipe=0 scale=micro rep=17/20 Y=0.2131
recipe=0 scale=micro rep=18/20 Y=0.2146
recipe=0 scale=micro rep=19/20 Y=0.2124
recipe=0 scale=micro rep=20/20 Y=0.1697
recipe=0 scale=bench rep=1/10 Y=7.5280
recipe=0 scale=bench rep=2/10 Y=7.4816
recipe=0 scale=bench rep=3/10 Y=7.6899
recipe=0 scale=bench rep=4/10 Y=7.6331
recipe=0 scale=bench rep=5/10

In [6]:
import pandas as pd
df = pd.read_csv("data/noise_bias.csv")
summary = df.groupby(["recipe_id", "scale"])["Y"].agg(["mean", "std", "count"])
summary["cv"] = summary["std"] / summary["mean"]  # relative noise
print(summary)

FileNotFoundError: [Errno 2] No such file or directory: 'data/noise_bias.csv'